In [4]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone

ODDS_API_KEY = os.getenv("THE_ODDS_API_KEY")

SPORT_KEY = "baseball_mlb"
REGIONS = 'us,us2'
MARKETS = "h2h"
ODDS_FORMAT = "decimal"

valid_bookmakers = set(['DraftKings', 'BetMGM', 'Caesars', 'FanDuel', 'theScore Bet'])

def decimal_to_implied_prob(decimal_odds):
    return 1 / decimal_odds


def fetch_live_odds_snapshot(
    sport_key: str,
    home_team: str,
    away_team: str,
) -> pd.DataFrame:
    url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds"

    params = {
        "apiKey": ODDS_API_KEY,
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": ODDS_FORMAT,
        "dateFormat": "iso",
    }

    resp = requests.get(url, params=params, timeout=20)
    resp.raise_for_status()

    snapshot_time = pd.Timestamp.now(tz="UTC")
    data = resp.json()

    rows = []

    for game in data:
        for bookmaker in game.get("bookmakers", []):
            if bookmaker['title'] not in valid_bookmakers:
                continue
            book = bookmaker["title"]
            book_key = bookmaker["key"]
            book_last_update = bookmaker.get("last_update")

            for market in bookmaker.get("markets", []):
                if market["key"] != "h2h":
                    continue

                for outcome in market.get("outcomes", []):
                    if outcome["name"] == home_team:
                        raw_odds = outcome["price"]

                        rows.append({
                            "snapshot_time": snapshot_time,
                            "book": book,
                            "book_key": book_key,
                            "book_last_update": book_last_update,
                            "event_id": game["id"],
                            "commence_time": game["commence_time"],
                            "home_team": game["home_team"],
                            "away_team": game["away_team"],
                            "target_team": home_team,
                            "raw_odds": raw_odds,
                            "implied_prob_raw": decimal_to_implied_prob(raw_odds),
                        })
                    elif outcome["name"] == away_team:
                        raw_odds = outcome["price"]

                        rows.append({
                            "snapshot_time": snapshot_time,
                            "book": book,
                            "book_key": book_key,
                            "book_last_update": book_last_update,
                            "event_id": game["id"],
                            "commence_time": game["commence_time"],
                            "home_team": game["home_team"],
                            "away_team": game["away_team"],
                            "target_team": away_team,
                            "raw_odds": raw_odds,
                            "implied_prob_raw": decimal_to_implied_prob(raw_odds),
                        })
    return pd.DataFrame(rows)

In [7]:
odds_df = fetch_live_odds_snapshot(
    sport_key="baseball_mlb",
    home_team="Philadelphia Phillies",
    away_team="San Diego Padres",
)

odds_df

HTTPError: 401 Client Error: Unauthorized for url: https://api.the-odds-api.com/v4/sports/baseball_mlb/odds?apiKey=c146d41ee6fa4c4af149f4965849f905&regions=us%2Cus2&markets=h2h&oddsFormat=decimal&dateFormat=iso

In [49]:
def poll_live_game_odds(
    sport_key: str,
    home_team: str,
    away_team: str,
    output_path: str = "data/kxmlbgame-26may251540nyykc.csv",
    poll_seconds: int = 60,
):
    while True:
        try:
            snap = fetch_live_odds_snapshot(
                sport_key=sport_key,
                home_team=home_team,
                away_team=away_team,
            )

            if not snap.empty:
                write_header = not os.path.exists(output_path)
                snap.to_csv(output_path, mode="a", header=write_header, index=False)

                print(
                    f"{pd.Timestamp.now(tz='UTC')} | saved {len(snap)} sportsbook rows"
                )
            else:
                print(f"{pd.Timestamp.now(tz='UTC')} | no matching game found")

        except Exception as e:
            print(f"Error: {e}")

        time.sleep(poll_seconds)

In [ ]:
poll_live_game_odds(
    sport_key="baseball_mlb",
    home_team="New York Yankees",
    away_team="Kansas City Royals",
    poll_seconds=5,
)

2026-05-25 20:54:56.483590+00:00 | saved 8 sportsbook rows
2026-05-25 20:54:57.240461+00:00 | saved 8 sportsbook rows
2026-05-25 20:54:57.999658+00:00 | saved 8 sportsbook rows
2026-05-25 20:54:58.766333+00:00 | saved 8 sportsbook rows
2026-05-25 20:54:59.523954+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:00.302674+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:01.094842+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:01.848453+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:02.605569+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:03.380895+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:04.144772+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:04.888469+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:05.654702+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:06.413834+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:07.182401+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:07.940959+00:00 | saved 8 sportsbook rows
2026-05-25 20:55:08.700959+00:00 | saved 8 sportsbook ro

KeyboardInterrupt: 